# N110 · 换根DP与全节点答案

**目标：** 从一个根的统计量推导相邻根的答案。

**先修：** N109, N075, N081。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 子树大小；两遍DFS；内外贡献；重根转移；距离和。

## 从问题到算法

只算一个根的答案不难，重复n遍却浪费了相邻根之间的大量共享信息。从u移根到孩子v，v子树内的size[v]个点距离减1，其余n−size[v]个点距离加1。方向猜测也是局部变化：只有跨越的那条边翻转，其他父子方向不变。

## 最小实现

**本章接口：** `sum_of_distances_in_tree(n, edges)`、`root_count(edges, guesses, k)`

In [1]:
def _tree_order(adj,root=0):
    if not adj: return [],[]
    parent=[-1]*len(adj); parent[root]=root; order=[root]
    for u in order:
        for v in adj[u]:
            if v==parent[u]: continue
            if parent[v]!=-1: raise ValueError('tree required')
            parent[v]=u; order.append(v)
    if len(order)!=len(adj): raise ValueError('connected tree required')
    return parent,order

def _adj_from_edges(n,edges):
    adj=[[] for _ in range(n)]
    for u,v in edges: adj[u].append(v); adj[v].append(u)
    return adj

def sum_of_distances_in_tree(n,edges):
    if not n: return []
    adj=_adj_from_edges(n,edges); parent,order=_tree_order(adj); size=[1]*n; depth=[0]*n
    for u in order[1:]: depth[u]=depth[parent[u]]+1
    for u in reversed(order[1:]): size[parent[u]]+=size[u]
    answer=[0]*n; answer[0]=sum(depth)
    for u in order[1:]: answer[u]=answer[parent[u]]+n-2*size[u]
    return answer

def root_count(edges,guesses,k):
    n=len(edges)+1; adj=_adj_from_edges(n,edges); parent,order=_tree_order(adj); guesses=set(map(tuple,guesses))
    score=[0]*n; score[0]=sum((parent[u],u) in guesses for u in order[1:])
    for u in order[1:]:
        p=parent[u]; score[u]=score[p]-((p,u) in guesses)+((u,p) in guesses)
    return sum(x>=k for x in score)

## 正确性、前提与复杂度

删除边(u,v)将树分成互补两部分，任意点到两个端点的路径必须经过其中一个端点，故距离变化恰为±1。先算根0，再沿树边传播的归纳覆盖全部根。方向得分只扣除旧方向并加入新方向。

**代价：** O(n+|guesses|)平均时间和O(n+|guesses|)空间。guesses按集合解释；编号0..n−1，无向边构成树。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1),(1,2),(1,3),(3,4)]; answer=sum_of_distances_in_tree(5,edges)
show_table(['根','距离总和'],list(enumerate(answer)))

根,距离总和
0,8
1,5
2,8
3,6
4,9


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert sum_of_distances_in_tree(3,[(0,1),(1,2)])==[3,2,3]
assert sum_of_distances_in_tree(1,[])==[0]
from random import Random
from collections import deque
rng=Random(110)
for n in range(1,16):
    edges=[(rng.randrange(v),v) for v in range(1,n)]; adj=_adj_from_edges(n,edges)
    guesses=[(v,u) if rng.randrange(2) else (u,v) for u,v in edges]; reference=[]; scores=[]
    for root in range(n):
        d=[-1]*n; d[root]=0; q=deque([root]); pairs=set()
        while q:
            u=q.popleft()
            for v in adj[u]:
                if d[v]==-1: d[v]=d[u]+1; q.append(v); pairs.add((u,v))
        reference.append(sum(d)); scores.append(len(pairs&set(guesses)))
    assert sum_of_distances_in_tree(n,edges)==reference
    for k in range(n+1): assert root_count(edges,guesses,k)==sum(x>=k for x in scores)
print('N110: 所有本章断言通过')

N110: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 推导ans[v]=ans[u]+n-2*size[v]。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 构造链和星形的闭式答案。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 834. Sum of Distances in Tree（canonical）
- 2581. Count Number of Possible Root Nodes（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。